# z009 - Juntar BAJA+1 y BAJA+2 como positivos

**Idea:** entrenar con 1 = BAJA+1 o BAJA+2 (el doble de ejemplos de "se está yendo").
La **ganancia se calcula solo con BAJA+2** (los BAJA+1 elegidos cuestan 27.500, como cualquier no BAJA+2).
Se corta por cantidad (top N), no por probabilidad.

Trial 34 de z005, 538 árboles, 5 semillas, train 202104, valid 202106.
Ojo: los hiperparámetros se ajustaron con solo BAJA+2 → si empata, valdría la pena rehacer Optuna.

In [1]:
# --- 1. Librerías y parámetros comunes ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MES_VALID = 202106
SEMILLAS  = [501419, 623353, 786349, 834569, 946331]
N_PARTICIONES = 50        # Shuffle Split del test: particiones al azar del 50% de 202106

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2. Cargar datos y lags (igual que z003) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria", "clase_binaria2"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
valid = df[df["foto_mes"] == MES_VALID]
# La ganancia SIEMPRE se calcula con BAJA+2 (aunque se entrene con otro target)
ganancia_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
print("Variables:", len(variables))

Variables: 761


In [3]:
# --- 3. Funciones de evaluación ---
def metricas(prob, mascara=None):
    # Ganancia acumulada ordenando de mayor a menor probabilidad (opcional: solo una parte de los clientes)
    g = ganancia_cliente if mascara is None else ganancia_cliente[mascara]
    p = prob if mascara is None else prob[mascara]
    acum = np.cumsum(g[np.argsort(p)[::-1]])
    n = int(np.argmax(acum)) + 1
    return acum[n - 1] / 1e6, n, acum[min(11_999, len(acum) - 1)] / 1e6

def resumen(probs_por_version):
    """probs_por_version = {version: [prob_semilla1, ..., prob_semilla5]}"""
    # a) Por semilla: ganancia máxima, mejor N y top 12.000
    filas = []
    for v, lista in probs_por_version.items():
        for s, pr in zip(SEMILLAS, lista):
            g, n, g12 = metricas(pr)
            filas.append({"version": v, "semilla": s, "ganancia_max_M": round(g, 1), "mejor_n": n, "top12000_M": round(g12, 1)})
    por_semilla = pd.DataFrame(filas)
    tabla = por_semilla.groupby("version", sort=False)[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1)

    # b) Shuffle Split del test: ensamble de las 5 semillas evaluado en 50 particiones del 50% de 202106
    rng = np.random.default_rng(SEMILLAS[0])
    ens = {v: np.mean(lista, axis=0) for v, lista in probs_por_version.items()}
    part = []
    for i in range(N_PARTICIONES):
        mascara = rng.random(len(ganancia_cliente)) < 0.5
        part.append({v: metricas(pr, mascara)[0] for v, pr in ens.items()})
    part = pd.DataFrame(part)
    tabla[("shuffle_split", "mean")] = part.mean().round(1)
    tabla[("shuffle_split", "std")] = part.std().round(1)
    tabla[("shuffle_split", "veces_mejor")] = part.idxmax(axis=1).value_counts().reindex(tabla.index).fillna(0).astype(int)
    tabla[("ensamble_5", "ganancia_max_M")] = [round(metricas(ens[v])[0], 1) for v in tabla.index]
    return por_semilla, tabla

def anotar(tabla, prefijo, meses_train, extra=""):
    ruta_res = "resultados_experimentos.csv"
    res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
    res = res[~res["descripcion"].str.startswith(prefijo)]   # sin duplicar si se vuelve a correr
    for v, r in tabla.iterrows():
        nueva = {
            "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
            "descripcion": (f"{prefijo} {v}: prom. {len(SEMILLAS)} semillas; shuffle split {r[('shuffle_split', 'mean')]} "
                            f"(mejor en {r[('shuffle_split', 'veces_mejor')]}/{N_PARTICIONES}){extra}"),
            "meses_train": str(meses_train), "mes_valid": MES_VALID, "semilla": str(SEMILLAS),
            "params": str(params), "mejor_n": int(r[("mejor_n", "mean")]),
            "ganancia_max_M": r[("ganancia_max_M", "mean")], "ganancia_top12000_M": r[("top12000_M", "mean")],
        }
        res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)
    res["n_enviados"] = res["n_enviados"].astype("Int64")
    res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
    print("Anotado en", ruta_res)

In [4]:
# --- 4. Entrenar las dos versiones ---
df["clase_binaria2"] = df["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)
train = df[df["foto_mes"] == 202104]
print("Positivos en train: solo BAJA+2 =", train["clase_binaria"].sum(), "| BAJA+1 + BAJA+2 =", train["clase_binaria2"].sum())

probs = {}
for version, target in [("Solo BAJA+2", "clase_binaria"), ("BAJA+1 + BAJA+2", "clase_binaria2")]:
    datos = lgb.Dataset(train[variables], label=train[target])
    probs[version] = [lgb.train({**params, "seed": s}, datos, num_boost_round=ARBOLES).predict(valid[variables])
                      for s in SEMILLAS]
    print(version, "listo")

Positivos en train: solo BAJA+2 = 1139 | BAJA+1 + BAJA+2 = 2103


Solo BAJA+2 listo


BAJA+1 + BAJA+2 listo


In [5]:
# --- 5. Comparación ---
por_semilla, tabla = resumen(probs)
print(por_semilla.pivot(index="semilla", columns="version", values="ganancia_max_M"))
tabla

version  BAJA+1 + BAJA+2  Solo BAJA+2
semilla                              
501419             427.1        400.5
623353             420.4        397.2
786349             421.5        401.4
834569             420.3        397.2
946331             419.8        401.8


ganancia_max_M      top12000_M       mejor_n          \
                          mean  std       mean  std     mean     std   
version                                                                
Solo BAJA+2              399.6  2.3      392.9  3.7  13212.0  1072.1   
BAJA+1 + BAJA+2          421.8  3.0      390.7  5.6  14700.8   353.3   

                shuffle_split                       ensamble_5  
                         mean   std veces_mejor ganancia_max_M  
version                                                         
Solo BAJA+2             199.1  11.6           1          402.4  
BAJA+1 + BAJA+2         209.5  13.7          49          422.7

In [6]:
# --- 6. ¿Cuántos BAJA+1 elige cada versión en su top 12.000? ---
clases = valid["clase_ternaria"].values
for v, lista in probs.items():
    top = np.argsort(np.mean(lista, axis=0))[::-1][:12_000]
    print(v, pd.Series(clases[top]).value_counts().to_dict())

Solo BAJA+2 {'CONTINUA': 10739, 'BAJA+2': 661, 'BAJA+1': 600}
BAJA+1 + BAJA+2 {'CONTINUA': 10717, 'BAJA+2': 657, 'BAJA+1': 626}


In [7]:
# --- 7. Anotar ---
anotar(tabla, "z009 juntar bajas", [202104])

Anotado en resultados_experimentos.csv


## Conclusiones (08/10)
- **BAJA+1 + BAJA+2: 421,8 ± 3,0 M** vs solo BAJA+2: 399,6 ± 2,3 → **+22 M**; Shuffle Split: mejor en 49 de 50.
- Pero el **mejor N sube a ~14.700** (vs ~13.200): en top 12.000 da 390,7 vs 392,9 (empate). Si se adopta, hay que mandar más clientes.
- Los hiperparámetros eran de solo BAJA+2: con Optuna sobre este target podría mejorar más.
- Decisión: pendiente (ver con Keila).